In [1]:
import open3d as o3d
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("../outputs")

# Load the merged + globally bundle adjusted point cloud from incremental_sfm.ipynb
pcd = o3d.io.read_point_cloud(str(OUTPUT_DIR / "merged_global_ba_reconstruction.ply"))
points = np.asarray(pcd.points)

# Per-point camera viewpoints (same point order as the .ply), used to orient the normals
viewpoints = np.load(OUTPUT_DIR / "merged_global_ba_viewpoints.npz")
view_centers = viewpoints["view_centers"]
assert len(view_centers) == len(points), "Viewpoints do not match the .ply, re-run cell 12 of incremental_sfm.ipynb"

print(f"Loaded {len(points)} 3D points (colors: {pcd.has_colors()}) and {len(viewpoints['camera_centers'])} cameras.")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded 20259 3D points (colors: True) and 135 cameras.


In [2]:
# 1. Estimate unoriented surface normals via Local PCA (k-NN Search)
# SfM has no metric scale, so the search radius is set relative to the typical point spacing.
# The radius must be larger than the remaining noise across the walls, otherwise the fitted
# planes (and normals) follow the noise: 25x spacing, and enough neighbours to fill that radius.
nn_spacing = float(np.median(pcd.compute_nearest_neighbor_distance()))
normal_radius = 25.0 * nn_spacing
print(f"Median point spacing: {nn_spacing:.4f} -> normal search radius: {normal_radius:.4f}")

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=normal_radius, max_nn=200))

# 2. Orient every normal towards the cameras that observed the point (outward facing surface)
normals = np.asarray(pcd.normals)
flip = np.einsum("ij,ij->i", normals, view_centers - points) < 0
normals[flip] *= -1
pcd.normals = o3d.utility.Vector3dVector(normals)
print(f"Flipped {flip.sum()} / {len(flip)} normals to face their cameras.")

# 3. Optional: Inspect oriented point cloud with normal vectors
o3d.visualization.draw_geometries([pcd], point_show_normal=True)

Median point spacing: 0.0181 -> normal search radius: 0.4527
Flipped 8982 / 20259 normals to face their cameras.


In [3]:
# 1. Run Screened Poisson Surface Reconstruction
# depth controls the octree resolution (depth=8 to 10 is standard).
# depth 8 = coarser grid -> smoother surface; finer detail than the point noise is not real anyway.
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=8, scale=1.1
)
print(f"Raw Mesh: {len(mesh.vertices)} vertices | {len(mesh.triangles)} triangles")

# 2. Trim low-density boundary artifacts ("floaters")
# Poisson builds a watertight surface. Regions where no 3D points existed
# (like empty sky/edges) get low-density "ghost" triangles that must be cropped.
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.05)  # remove bottom 5% lowest density
mesh.remove_vertices_by_mask(densities < density_threshold)
print(f"Trimmed Mesh: {len(mesh.vertices)} vertices | {len(mesh.triangles)} triangles remaining")

# 3. Compute vertex normals for smooth shading and export the mesh (vertex colors come from the point cloud)
mesh.compute_vertex_normals()
mesh_path = OUTPUT_DIR / "building_mesh.ply"
if o3d.io.write_triangle_mesh(str(mesh_path), mesh):
    print(f"Saved 3D mesh ({len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles) to: {mesh_path.resolve()}")

Raw Mesh: 37288 vertices | 74316 triangles
Trimmed Mesh: 35423 vertices | 69978 triangles remaining
Saved 3D mesh (35423 vertices, 69978 triangles) to: /home/semi/projects/propyläen_3d_reconstruct/outputs/building_mesh.ply


In [4]:
# Render the interactive 3D surface mesh
o3d.visualization.draw_geometries(
    [mesh],
    window_name="Reconstructed 3D Surface Mesh",
    width=1024,
    height=768,
    mesh_show_wireframe=False,
    mesh_show_back_face=True,
)


(python:132072): Gtk-WARNING **: 00:50:56.476: gtk_disable_setlocale() must be called before gtk_init()
